# **Ejemplo completo de KMeans con dataset real (Wine dataset) usando sklearn**

**1.Importar librerías necesarias**



In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans                 #Clase que implementa KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score       # Metrica de evaluación Clusters
import seaborn as sns

# Estilo para gráficos
sns.set(style="whitegrid")


🍷 **2.Cargar el dataset del vino**

In [ ]:

wine = load_wine()
X = wine.data
y = wine.target
features = wine.feature_names

print(f"Forma de los datos: {X.shape}")
df = pd.DataFrame(X, columns=features)
df.head()


🔧 **3. Estandarizar los datos**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

📊 **4. Método del Codo para elegir K**

los parámetros utilizados para **KMeans** son:

* `n_clusters`: Número de clústeres (grupos) que quieres que el algoritmo forme. Por ejemplo, k=3 buscará agrupar los datos en 3 grupos.
* `n_init`: Número de veces que el algoritmo se reinicializa con diferentes centroides aleatorios. Al final, selecciona la mejor agrupación (la que tenga menor inercia). Valor por defecto (desde scikit-learn v1.2 en adelante) es 'auto', pero tradicionalmente se usaba 10.

El atributo `kmeans.inertia_` en scikit-learn representa una métrica de la calidad del agrupamiento hecha por el algoritmo K-Means.

In [ ]:
inertia = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)   # Métrica Inercia.

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker='o')
plt.title('Método del Codo')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia')
plt.xticks(K_range)
plt.grid(True)
plt.show()

🧭 **5. Silhouette Score para validar K**

La función `silhouette_score` de `sklearn.metrics`, se usa para evaluar la calidad de un agrupamiento, especialmente útil en algoritmos como **K-Means** y **DBSCAN**.

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='green')
plt.title('Silhouette Score por número de clusters')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

✅ **6. Aplicar KMeans con K=3**

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

**Ver las distancias a los centroides**

In [ ]:
from scipy.spatial.distance import cdist

# Calculamos la distancia de cada punto a cada centroide
distancias = cdist(X_scaled, kmeans.cluster_centers_, metric='euclidean')

# Distancia del primer punto a todos los centroides
print("Distancia del primer vino a cada centroide:")
print(distancias[0])

# También podrías visualizar cómo se ve la matriz de distancias
dist_df = pd.DataFrame(distancias, columns=[f"Centroide {i}" for i in range(optimal_k)])
dist_df.head()

7. **Visualización con PCA**

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for i in range(optimal_k):
    plt.scatter(X_pca[clusters == i, 0], X_pca[clusters == i, 1], label=f'Cluster {i}')

plt.scatter(pca.transform(kmeans.cluster_centers_)[:, 0],
            pca.transform(kmeans.cluster_centers_)[:, 1],
            s=250, c='black', marker='X', label='Centroides')
plt.title('Visualización de Clusters con PCA')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.grid(True)
plt.show()

**🎯 8. Comparar con las clases reales (opcional)**

In [ ]:
plt.figure(figsize=(8, 6))
for i in np.unique(y):
    plt.scatter(X_pca[y == i, 0], X_pca[y == i, 1], label=f'Clase real {i}')

plt.title('Distribución de las clases reales (Wine dataset)')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True)
plt.show()

**Predecir un caso de prueba manual**

In [ ]:
# Un nuevo vino con características ficticias
nuevo_vino = np.array([[13.0, 2.0, 2.4, 15.0, 100.0, 2.5, 2.5, 0.3, 1.6, 5.0, 1.0, 3.0, 1000.0]])
nuevo_vino_scaled = scaler.transform(nuevo_vino)

# Predecimos a qué cluster pertenecería
cluster_predicho = kmeans.predict(nuevo_vino_scaled)
print(f"Este nuevo vino pertenecería al cluster: {cluster_predicho[0]}")

In [ ]:
# Coordenadas PCA del nuevo vino
nuevo_pca = pca.transform(nuevo_vino_scaled)

# Volver a graficar los clusters + vino nuevo
plt.figure(figsize=(8, 6))
for i in range(optimal_k):
    plt.scatter(X_pca[clusters == i, 0], X_pca[clusters == i, 1], label=f'Cluster {i}')

plt.scatter(pca.transform(kmeans.cluster_centers_)[:, 0],
            pca.transform(kmeans.cluster_centers_)[:, 1],
            s=200, c='black', marker='X', label='Centroides')

plt.scatter(nuevo_pca[:, 0], nuevo_pca[:, 1], c='red', s=150, marker='*', label='Nuevo vino')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('Nuevo vino proyectado con los clusters')
plt.legend()
plt.grid(True)
plt.show()


**... el modelo plantea: "Basado en su composición química, es más parecido a los vinos del Cluster 2".**